In [1]:
import fastf1
import pandas as pd
import time
from pathlib import Path

CACHE_DIR  = Path(r"C:\f1 new")
OUTPUT_DIR = Path(r"C:\f1 new\data")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))

YEARS = [2022, 2023, 2024, 2025]

print(f"Cache  → {CACHE_DIR.resolve()}")
print(f"Output → {OUTPUT_DIR.resolve()}")

Cache  → C:\f1 new
Output → C:\f1 new\data


In [2]:
all_schedules = []

for year in YEARS:
    print(f"Fetching schedule for {year}...")
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)

        schedule['Year'] = int(year)
        all_schedules.append(schedule)
        print(f"  {year}: {len(schedule)} race events")

    except Exception as e:
        print(f"  Error for {year}: {e}")

all_schedules_df = pd.concat(all_schedules, ignore_index=True)


all_schedules_df['Year']        = all_schedules_df['Year'].astype(int)
all_schedules_df['RoundNumber'] = all_schedules_df['RoundNumber'].astype(int)

csv_path = OUTPUT_DIR / "f1_schedule_2022_2025.csv"
all_schedules_df.to_csv(csv_path, index=False)

print(f"\nTotal events: {len(all_schedules_df)}")
print(f"Saved → {csv_path}")
all_schedules_df[['Year', 'RoundNumber', 'Country', 'EventName', 'EventDate']].head(10)

Fetching schedule for 2022...


C:\Users\DELL\AppData\Local\Temp\ipykernel_29216\815767591.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schedule['Year'] = int(year)


  2022: 22 race events
Fetching schedule for 2023...


C:\Users\DELL\AppData\Local\Temp\ipykernel_29216\815767591.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schedule['Year'] = int(year)


  2023: 22 race events
Fetching schedule for 2024...
  2024: 24 race events
Fetching schedule for 2025...
  2025: 24 race events

Total events: 92
Saved → C:\f1 new\data\f1_schedule_2022_2025.csv


C:\Users\DELL\AppData\Local\Temp\ipykernel_29216\815767591.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schedule['Year'] = int(year)
C:\Users\DELL\AppData\Local\Temp\ipykernel_29216\815767591.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schedule['Year'] = int(year)


,Year,RoundNumber,Country,EventName,EventDate
0,2022,1,Bahrain,Bahrain Grand Prix,2022-03-20
1,2022,2,Saudi Arabia,Saudi Arabian Grand Prix,2022-03-27
2,2022,3,Australia,Australian Grand Prix,2022-04-10
3,2022,4,Italy,Emilia Romagna Grand Prix,2022-04-24
4,2022,5,United States,Miami Grand Prix,2022-05-08
5,2022,6,Spain,Spanish Grand Prix,2022-05-22
6,2022,7,Monaco,Monaco Grand Prix,2022-05-29
7,2022,8,Azerbaijan,Azerbaijan Grand Prix,2022-06-12
8,2022,9,Canada,Canadian Grand Prix,2022-06-19
9,2022,10,Great Britain,British Grand Prix,2022-07-03


In [3]:
print("Columns:", all_schedules_df.columns.tolist())
print()
print("Events per year:")
print(all_schedules_df.groupby('Year').size().to_string())
print()
print("dtypes check:")
print(all_schedules_df[['Year', 'RoundNumber']].dtypes)

Columns: ['RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventDate', 'EventName', 'EventFormat', 'Session1', 'Session1Date', 'Session1DateUtc', 'Session2', 'Session2Date', 'Session2DateUtc', 'Session3', 'Session3Date', 'Session3DateUtc', 'Session4', 'Session4Date', 'Session4DateUtc', 'Session5', 'Session5Date', 'Session5DateUtc', 'F1ApiSupport', 'Year']

Events per year:
Year
2022    22
2023    22
2024    24
2025    24

dtypes check:
Year           int64
RoundNumber    int64
dtype: object


In [4]:
schedule_df = pd.read_csv(OUTPUT_DIR / "f1_schedule_2022_2025.csv")

test_sample = schedule_df[
    (schedule_df['Year'] == 2024) &
    (schedule_df['RoundNumber'] <= 3)
]

test_laps = []

for _, row in test_sample.iterrows():
    year      = int(row['Year'])
    round_num = int(row['RoundNumber'])
    country   = row['Country']

    print(f"Fetching: {year} | R{round_num:02d} | {country}")

    try:
        session = fastf1.get_session(year, round_num, 'R')
        session.load(telemetry=False, weather=True)

        laps = session.laps.copy()
        laps['Year']        = year
        laps['RoundNumber'] = round_num
        laps['Country']     = country
        laps['SessionType'] = 'R'

        test_laps.append(laps)
        print(f"{len(laps)} laps loaded")

    except Exception as e:
        print(f" Error: {e}")

    time.sleep(2)

if test_laps:
    test_df = pd.concat(test_laps, ignore_index=True)
    print(f"\nTest complete — {len(test_df)} total laps")
    display(test_df[['Country', 'Year', 'RoundNumber', 'Driver',
                    'LapNumber', 'LapTime', 'Compound', 'TyreLife']].head(10))

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]


Fetching: 2024 | R01 | Bahrain


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


1129 laps loaded


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Fetching: 2024 | R03 | Australia


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']


998 laps loaded

Test complete — 3028 total laps


,Country,Year,RoundNumber,Driver,LapNumber,LapTime,Compound,TyreLife
0,Bahrain,2024,1,VER,1.0,0 days 00:01:37.284000,SOFT,4.0
1,Bahrain,2024,1,VER,2.0,0 days 00:01:36.296000,SOFT,5.0
2,Bahrain,2024,1,VER,3.0,0 days 00:01:36.753000,SOFT,6.0
3,Bahrain,2024,1,VER,4.0,0 days 00:01:36.647000,SOFT,7.0
4,Bahrain,2024,1,VER,5.0,0 days 00:01:37.173000,SOFT,8.0
5,Bahrain,2024,1,VER,6.0,0 days 00:01:37.092000,SOFT,9.0
6,Bahrain,2024,1,VER,7.0,0 days 00:01:37.038000,SOFT,10.0
7,Bahrain,2024,1,VER,8.0,0 days 00:01:37.024000,SOFT,11.0
8,Bahrain,2024,1,VER,9.0,0 days 00:01:37.229000,SOFT,12.0
9,Bahrain,2024,1,VER,10.0,0 days 00:01:36.960000,SOFT,13.0


In [5]:
all_laps    = []
all_results = []  

for _, row in schedule_df.iterrows():
    year       = int(row['Year'])
    round_num  = int(row['RoundNumber'])
    country    = str(row['Country'])
    event_name = str(row['EventName'])

    print(f"{year} | R{round_num:02d} | {country}")

    try:
        session = fastf1.get_session(year, round_num, 'R')
        session.load(telemetry=False, weather=True)

        laps    = session.laps.copy()
        weather = session.weather_data.copy()

        laps = pd.merge_asof(
            laps.sort_values('Time'),
            weather[['Time', 'AirTemp', 'TrackTemp',
                    'Humidity', 'WindSpeed', 'Rainfall']].sort_values('Time'),
            on='Time',
            direction='backward'
        )

        laps['Year']        = year
        laps['RoundNumber'] = round_num
        laps['Country']     = country
        laps['EventName']   = event_name
        laps['SessionType'] = 'R'
        all_laps.append(laps)

        if session.results is not None and not session.results.empty:
            results = session.results.copy()
            results['Year']        = year
            results['RoundNumber'] = round_num
            results['Country']     = country
            results['EventName']   = event_name
            all_results.append(results)

        print(f"  {len(laps)} laps")

    except Exception as e:
        print(f"  Error: {e}")
        time.sleep(2)
        continue

    time.sleep(5)

laps_df = pd.concat(all_laps, ignore_index=True)
laps_df.to_parquet(OUTPUT_DIR / "f1_laps_2022_2025.parquet", index=False, compression='snappy')
print(f"Laps saved: {len(laps_df):,} rows")

results_df = pd.concat(all_results, ignore_index=True)   
results_df.to_parquet(OUTPUT_DIR / "f1_results_2022_2025.parquet", index=False, compression='snappy')
print(f"Results saved: {len(results_df):,} rows")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


2022 | R01 | Bahrain


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


  1125 laps


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...

  1067 laps


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


2025 | R24 | United Arab Emirates


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']


  1156 laps
Laps saved: 101,297 rows
Results saved: 1,838 rows


In [6]:
import fastf1
session = fastf1.get_session(2024, 1, 'R')
session.load(telemetry=False, weather=False)
print(session.race_control_messages.head())

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


                 Time Category                                  Message  \
0 2024-03-02 14:06:41    Other  PINK HEAD PADDING MATERIAL MUST BE USED   
1 2024-03-02 14:20:01     Flag              GREEN LIGHT - PIT EXIT OPEN   
2 2024-03-02 14:30:01    Other                          PIT EXIT CLOSED   
3 2024-03-02 14:45:29    Other           RISK OF RAIN FOR F1 RACE IS 0%   
4 2024-03-02 14:57:04      Drs                             DRS DISABLED   

     Status   Flag  Scope  Sector RacingNumber  Lap  
0      None   None   None     NaN         None    1  
1      None  GREEN  Track     NaN         None    1  
2      None   None   None     NaN         None    1  
3      None   None   None     NaN         None    1  
4  DISABLED   None   None     NaN         None    1  


In [7]:
all_rc = []

for _, row in schedule_df.iterrows():
    year      = int(row['Year'])
    round_num = int(row['RoundNumber'])
    country   = str(row['Country'])

    try:
        session = fastf1.get_session(year, round_num, 'R')
        session.load(telemetry=False, weather=False)  

        if session.race_control_messages is not None:
            rc = session.race_control_messages.copy()
            rc['Year']        = year
            rc['RoundNumber'] = round_num
            rc['Country']     = country
            all_rc.append(rc)
            print(f"{year} | R{round_num:02d} | {len(rc)} messages")

    except Exception as e:
        print(f"{year} | R{round_num:02d} | {e}")
        continue

    time.sleep(1) 

rc_df = pd.concat(all_rc, ignore_index=True)
rc_df.to_parquet(OUTPUT_DIR / "f1_race_control.parquet",
                index=False, compression='snappy')

print(f"\nRC saved: {len(rc_df):,} messages")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


2022 | R01 | 60 messages


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 22
core        WARNING 	No lap data for driver 47
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4',

2022 | R02 | 90 messages


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.140000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '11', '63', '44', '4', '3', '31', '77', '10', '23', '24', '18', '47', '20', '22', '6', '14', '1', '5', '55']


...

core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '55', '4', '12', '63', '14', '16', '30', '22', '23', '44', '5', '43', '31', '10', '18', '6', '87', '27']


2025 | R23 | 79 messages


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']


2025 | R24 | 107 messages

RC saved: 8,156 messages


C:\Users\DELL\AppData\Local\Temp\ipykernel_29216\18235366.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rc_df = pd.concat(all_rc, ignore_index=True)


In [ ]:
rc_df = pd.read_parquet(OUTPUT_DIR / "f1_race_control.parquet")

sc_df = rc_df[rc_df['Message'].str.contains(
    'SAFETY CAR DEPLOYED|VIRTUAL SAFETY CAR DEPLOYED', na=False
)]

print(f"SC Events: {len(sc_df)}")
print(sc_df[['Year', 'RoundNumber', 'Message', 'Lap']].head(10))

SC Events: 121
     Year  RoundNumber                      Message   Lap
47   2022            1  VIRTUAL SAFETY CAR DEPLOYED  46.0
48   2022            1          SAFETY CAR DEPLOYED  46.0
97   2022            2  VIRTUAL SAFETY CAR DEPLOYED  16.0
98   2022            2          SAFETY CAR DEPLOYED  16.0
119  2022            2  VIRTUAL SAFETY CAR DEPLOYED  38.0
156  2022            3  VIRTUAL SAFETY CAR DEPLOYED   3.0
157  2022            3          SAFETY CAR DEPLOYED   3.0
173  2022            3          SAFETY CAR DEPLOYED  23.0
181  2022            3  VIRTUAL SAFETY CAR DEPLOYED  39.0
223  2022            4          SAFETY CAR DEPLOYED   1.0


: 